# WP2: Pre-processing & Harmonisation

Apply cloud masking and NDSI computation to Sentinel-2; apply speckle filtering to Sentinel-1.

In [ ]:
import ee
import geemap
import sys
sys.path.insert(0, '..')
from src.utils import load_aoi, get_gee_project
from src.preprocessing import preprocess_s2, preprocess_s1

ee.Initialize(project=get_gee_project())
aoi = load_aoi()

## 2.1 Sentinel-2: cloud masking + NDSI

In [ ]:
START, END = '2019-01-01', '2024-12-31'

s2_raw = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate(START, END)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 80))
)

s2_processed = s2_raw.map(preprocess_s2)
print('Processed S2 images:', s2_processed.size().getInfo())

## 2.2 Sentinel-1: speckle filtering

In [ ]:
s1_raw = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filterDate(START, END)
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .select(['VV', 'VH'])
)

s1_processed = s1_raw.map(preprocess_s1)
print('Processed S1 images:', s1_processed.size().getInfo())

## 2.3 Inspect a single image

In [ ]:
sample_s2 = s2_processed.first()
sample_s1 = s1_processed.first()

Map = geemap.Map()
Map.centerObject(aoi, zoom=9)
Map.addLayer(sample_s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}, 'S2 RGB')
Map.addLayer(sample_s2.select('NDSI'), {'min': -1, 'max': 1, 'palette': ['blue', 'white']}, 'NDSI')
Map.addLayer(sample_s1.select('VV'), {'min': -25, 'max': 0}, 'S1 VV')
Map